# Multi-State Batch Generation and Dataset Export

When building a robust evaluation suite for LLM reasoning on government benefits, a single state or program is rarely enough. This notebook shows how to:

- Generate large, diverse test suites spanning multiple states and programs with `BatchPipeline`
- Analyze the resulting case distribution (eligibility splits, difficulty, variation tags)
- Export to JSONL for fine-tuning pipelines and HuggingFace Dataset format for `push_to_hub()`
- Verify that `seed=42` produces bit-identical case IDs on re-run

Cross-state variation matters because states implement the same federal programs differently. Virginia applies the strict federal asset test for SNAP; California and Maryland use Broad-Based Categorical Eligibility (BBCE) which removes the asset test entirely; Texas enforces strict limits and does not expand Medicaid. A model that can reason correctly only for one state's rules is not robust.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
from collections import Counter

from govsynth import Pipeline, BatchPipeline, list_presets

os.makedirs('./suite_output', exist_ok=True)
print('Ready.')

## 1. Available Presets

Each preset bundles a source connector, generator class, and default profile strategy. `list_presets()` prints the full registry — use these names as-is with `Pipeline.from_preset()` or `BatchPipeline.from_presets()`.

In [ ]:
list_presets()

Key policy differences across the SNAP state presets:

| Preset | Asset test | Notable rule |
|---|---|---|
| `snap.va` | Strict federal ($2,750 / $4,250 elderly) | Standard gross/net income limits |
| `snap.ca` | None (BBCE) | Auto-eligible if receiving other benefits |
| `snap.tx` | Strict federal | No Medicaid expansion; strict limits |
| `snap.md` | None (BBCE) | BBCE state similar to CA |

`wic.national` uses the single national 185% FPL income threshold regardless of state.

## 2. Single-Program Multi-State Batch — SNAP

Generate 30 SNAP cases each for Virginia, California, and Texas (90 total). The `BatchPipeline` assigns each pipeline `seed + i` so the three runs are independent but the overall batch is reproducible.

In [ ]:
snap_batch = BatchPipeline.from_presets(['snap.va', 'snap.ca', 'snap.tx'])
snap_cases = snap_batch.generate(n_per_pipeline=30, seed=42)

print(f'Total SNAP cases: {len(snap_cases)}')

In [ ]:
# Aggregate stats broken out by jurisdiction
print(f'{"Jurisdiction":<12} {"Cases":>6} {"Eligible":>9} {"Ineligible":>11} {"Elig %":>8}')
print('-' * 52)

for jurisdiction in sorted(set(c.jurisdiction for c in snap_cases)):
    subset = [c for c in snap_cases if c.jurisdiction == jurisdiction]
    n = len(subset)
    elig = sum(1 for c in subset if c.expected_outcome == 'eligible')
    inelig = n - elig
    pct = elig / n * 100 if n else 0
    print(f'{jurisdiction:<12} {n:>6} {elig:>9} {inelig:>11} {pct:>7.0f}%')

print()
diff_counts = Counter(c.difficulty.value for c in snap_cases)
print('Difficulty distribution:')
for d in ['easy', 'medium', 'hard', 'adversarial']:
    n = diff_counts.get(d, 0)
    bar = '█' * int(n / len(snap_cases) * 40)
    print(f'  {d:<12} {n:>3}  {bar}')

## 3. Multi-Program Batch — SNAP + WIC

Mixing programs in a single batch demonstrates how case IDs encode both the program and jurisdiction, keeping the pool unambiguous. WIC eligibility is based on categorical status (pregnant, postpartum, infant, child under 5) plus a national 185% FPL income limit — structurally different from SNAP's multi-test gross/net/asset chain.

In [ ]:
wic_batch = BatchPipeline.from_presets(['wic.national'])
wic_cases = wic_batch.generate(n_per_pipeline=30, seed=42)

all_cases = snap_cases + wic_cases
print(f'Total cases: {len(all_cases)}  '
      f'(SNAP: {len(snap_cases)}, WIC: {len(wic_cases)})')

In [ ]:
# Case IDs distinguish program and jurisdiction at a glance
print('Sample case IDs by program:')
print()

for program in ('snap', 'wic'):
    subset = [c for c in all_cases if c.program == program]
    print(f'  [{program.upper()}]')
    for case in subset[:4]:
        print(f'    {case.case_id}')
    print()

## 4. Dataset Statistics

With ~120 cases across two programs and three states, we can profile the pool before export to catch imbalances early.

In [ ]:
# Cases per jurisdiction
by_jurisdiction = Counter(c.jurisdiction for c in all_cases)
print(f'{"Jurisdiction":<14} {"Cases":>6}')
print('-' * 22)
for j, n in sorted(by_jurisdiction.items()):
    print(f'{j:<14} {n:>6}')

In [ ]:
# Eligible/ineligible split per program
print(f'{"Program":<8} {"Eligible":>9} {"Ineligible":>11} {"Total":>7} {"Elig %":>8}')
print('-' * 48)

for program in sorted(set(c.program for c in all_cases)):
    subset = [c for c in all_cases if c.program == program]
    elig = sum(1 for c in subset if c.expected_outcome == 'eligible')
    inelig = len(subset) - elig
    pct = elig / len(subset) * 100
    print(f'{program:<8} {elig:>9} {inelig:>11} {len(subset):>7} {pct:>7.0f}%')

In [ ]:
# Difficulty distribution across the full suite
by_difficulty = Counter(c.difficulty.value for c in all_cases)
total = len(all_cases)

print('Difficulty distribution:')
for d in ['easy', 'medium', 'hard', 'adversarial']:
    n = by_difficulty.get(d, 0)
    pct = n / total * 100 if total else 0
    bar = '█' * int(pct / 2)
    print(f'  {d:<12} {n:>4}  ({pct:>4.1f}%)  {bar}')

In [ ]:
# Variation tag frequency — shows which edge-case dimensions are covered
all_tags = [tag for case in all_cases for tag in case.variation_tags]
tag_counts = Counter(all_tags)

print(f'Top variation tags (across {len(all_cases)} cases):')
for tag, count in tag_counts.most_common(15):
    bar = '█' * int(count / total * 80)
    print(f'  {tag:<35} {count:>4}  {bar}')

## 5. Export to JSONL

JSONL is the standard format for fine-tuning and evaluation pipelines. Each line is a self-contained JSON object with the full case including the rationale trace.

In [ ]:
# Use Pipeline.save() to write JSONL — BatchPipeline.save() is deprecated.
# We instantiate any pipeline just to access the formatter.
pipeline = Pipeline.from_preset('snap.va')

pipeline.save(all_cases, './suite_output/multi_state_suite.jsonl', formats=['jsonl'])
print('Written.')

In [ ]:
# Inspect the first few lines
with open('./suite_output/multi_state_suite.jsonl') as f:
    lines = [json.loads(f.readline()) for _ in range(3)]

print(f'Top-level keys: {list(lines[0].keys())}')
print()

for i, record in enumerate(lines):
    print(f'--- record {i} ---')
    print(f'  case_id:    {record["case_id"]}')
    print(f'  program:    {record["metadata"]["program"]}')
    print(f'  difficulty: {record["metadata"]["difficulty"]}')
    print(f'  outcome:    {record["metadata"]["expected_outcome"]}')
    user_msg = next(m for m in record['messages'] if m['role'] == 'user')
    print(f'  question:   {user_msg["content"][:90]}...')
    print()

## 6. Export to HuggingFace Dataset Format

`HFDatasetFormatter` converts the case list to a `datasets.DatasetDict` with configurable train/validation/test splits. This is the format expected by `push_to_hub()`.

Install the optional dependency with:
```
pip install synthetic-gov-data-kit[hf]
```

In [ ]:
try:
    from govsynth.formatters.hf_dataset import HFDatasetFormatter

    formatter = HFDatasetFormatter(split_ratios={'train': 0.7, 'validation': 0.15, 'test': 0.15})
    ds = formatter.to_dataset(all_cases)

    print('DatasetDict splits:')
    for split_name, split_ds in ds.items():
        print(f'  {split_name:<12} {len(split_ds):>4} rows')

    print()
    print('Column names:', ds['train'].column_names)
    print()

    # Peek at one row
    row = ds['train'][0]
    print('Sample row fields:')
    for key in ['case_id', 'program', 'jurisdiction', 'difficulty', 'expected_outcome',
                'household_size', 'monthly_gross_income']:
        print(f'  {key:<25} {row[key]}')

    # Save to disk in Arrow format
    formatter.write(all_cases, './suite_output/hf_dataset/')
    print()
    print('Saved Arrow dataset → ./suite_output/hf_dataset/')
    print('Push to hub with: ds.push_to_hub("your-org/govsynth-multi-state")')

except ImportError:
    # datasets not installed — show how to load from JSONL instead
    print('HuggingFace datasets library not installed.')
    print('Install with: pip install synthetic-gov-data-kit[hf]')
    print()
    print('Alternatively, load from the JSONL file directly:')
    print()
    print('    from datasets import load_dataset')
    print('    ds = load_dataset(')
    print('        "json",')
    print('        data_files="./suite_output/multi_state_suite.jsonl",')
    print('        split="train",')
    print('    )')
    print()
    print('Or split manually:')
    print()
    print('    ds = ds.train_test_split(test_size=0.15, seed=42)')

## 7. Reproducibility

Every call to `generate()` with the same `seed` and `n` must return identical case IDs. This is required for evaluation benchmarks where the test set must be pinned across runs and institutions.

We verify by re-running the same batch and comparing case ID sets.

In [ ]:
# First run — already in snap_cases from above
ids_run1 = [c.case_id for c in snap_cases]

# Second run — identical parameters
snap_batch2 = BatchPipeline.from_presets(['snap.va', 'snap.ca', 'snap.tx'])
snap_cases2 = snap_batch2.generate(n_per_pipeline=30, seed=42)
ids_run2 = [c.case_id for c in snap_cases2]

# Compare
identical = ids_run1 == ids_run2
print(f'Run 1 case count: {len(ids_run1)}')
print(f'Run 2 case count: {len(ids_run2)}')
print(f'Identical order:  {identical}')

if not identical:
    diff = [(a, b) for a, b in zip(ids_run1, ids_run2) if a != b]
    print(f'First differing pair: {diff[0]}')

In [ ]:
# Confirm a different seed yields a different (but still reproducible) set
snap_batch3 = BatchPipeline.from_presets(['snap.va', 'snap.ca', 'snap.tx'])
snap_cases3 = snap_batch3.generate(n_per_pipeline=30, seed=99)
ids_run3 = [c.case_id for c in snap_cases3]

overlap = set(ids_run1) & set(ids_run3)
print(f'seed=42 IDs: {len(ids_run1)}')
print(f'seed=99 IDs: {len(ids_run3)}')
print(f'Shared IDs:  {len(overlap)}')
print()
print('seed=42 sample:', ids_run1[:2])
print('seed=99 sample:', ids_run3[:2])